# Kapitola 5: Formátování výstupu a mluvení za Claude

- [Lekce](#lesson)
- [Cvičení](#exercises)
- [Příkladové hřiště](#example-playground)

## Nastavení

Spusťte následující buňku nastavení pro načtení vašeho API klíče a vytvoření pomocné funkce `get_completion`.

In [ ]:
```python
# Import vestavěnou knihovnu regulárních výrazů v Pythonu
import re
import boto3
import json

# Import modulu hints z balíčku utils
import os
import sys
module_path = ".."
sys.path.append(os.path.abspath(module_path))
from utils import hints

# Načtení proměnné MODEL_NAME z úložiště IPython
%store -r MODEL_NAME
%store -r AWS_REGION

client = boto3.client('bedrock-runtime', region_name=AWS_REGION)

def get_completion(prompt, system='', prefill=''):
    body = json.dumps(
        {
            "anthropic_version": '',
            "max_tokens": 2000,
            "messages":[
              {"role": "user", "content": prompt},
              {"role": "assistant", "content": prefill}
            ],
            "temperature": 0.0,
            "top_p": 1,
            "system": system
        }
    )
    response = client.invoke_model(body=body, modelId=MODEL_NAME)
    response_body = json.loads(response.get('body').read())

    return response_body.get('content')[0].get('text')
```

---

## Lekce

**Claude může formátovat svůj výstup mnoha různými způsoby**. Stačí ho o to požádat!

Jedním z těchto způsobů je použití XML tagů k oddělení odpovědi od jakéhokoli jiného nadbytečného textu. Už jste se naučili, že můžete použít XML tagy, aby byl váš prompt pro Claude jasnější a lépe zpracovatelný. Ukazuje se, že můžete také požádat Claude, aby **použil XML tagy, aby jeho výstup byl jasnější a snadněji pochopitelný** pro lidi.

### Příklady

Pamatujete si na 'problém s úvodem básně', který jsme vyřešili v kapitole 2 tím, že jsme Claudovi řekli, aby úvod zcela přeskočil? Ukazuje se, že můžeme dosáhnout podobného výsledku také tím, že **řekneme Claudovi, aby báseň vložil do XML tagů**.

In [ ]:
```python
# Obsah proměnné
ANIMAL = "Rabbit"

# Šablona promptu s místem pro obsah proměnné
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Vytisknout odpověď od Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))
```

Proč bychom to chtěli udělat? No, mít výstup v **XML značkách umožňuje koncovému uživateli spolehlivě získat báseň a pouze báseň tím, že napíše krátký program pro extrakci obsahu mezi XML značkami**.

Rozšířením této techniky je **vložit první XML značku do `assistant` části. Když vložíte text do `assistant` části, v podstatě říkáte Claudovi, že Claude už něco řekl, a že by měl pokračovat od tohoto bodu dál. Tato technika se nazývá "mluvení za Clauda" nebo "předvyplnění Claudeovy odpovědi."

Níže jsme to udělali s první `<haiku>` XML značkou. Všimněte si, jak Claude pokračuje přímo od místa, kde jsme skončili.

In [ ]:
```python
# Obsah proměnné
ANIMAL = "Cat"

# Šablona promptu s místem pro obsah proměnné
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Předvyplnění pro odpověď Claude
PREFILL = "<haiku>"

# Výpis odpovědi Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN:")
print(PROMPT)
print("\nASSISTANT TURN:")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))
```

Claude také vyniká v používání jiných stylů formátování výstupu, zejména `JSON`. Pokud chcete vynutit výstup ve formátu JSON (ne deterministicky, ale blízko tomu), můžete také předvyplnit Claudeovu odpověď otevírací závorkou, `{`.

In [ ]:
```python
# Obsah proměnné
ANIMAL = "Cat"

# Šablona promptu s místem pro obsah proměnné
PROMPT = f"Please write a haiku about {ANIMAL}. Use JSON format with the keys as \"first_line\", \"second_line\", and \"third_line\"."

# Předvyplnění pro odpověď Claude
PREFILL = "{"

# Tisk odpovědi Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))
```

Níže je příklad **více vstupních proměnných ve stejném promptu A specifikace formátování výstupu, vše provedené pomocí XML tagů**.

In [ ]:
```python
# První vstupní proměnná
EMAIL = "Hi Zack, just pinging you for a quick update on that prompt you were supposed to write."

# Druhá vstupní proměnná
ADJECTIVE = "olde english"

# Šablona promptu s místem pro obsah proměnné
PROMPT = f"Hey Claude. Here is an email: <email>{EMAIL}</email>. Make this email more {ADJECTIVE}. Write the new version in <{ADJECTIVE}_email> XML tags."

# Předvyplnění pro Claudeovu odpověď (nyní jako f-string s proměnnou)
PREFILL = f"<{ADJECTIVE}_email>"

# Výpis Claudeovy odpovědi
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))
```

#### Bonusová lekce

Pokud voláte Claude přes API, můžete předat uzavírací XML tag parametru `stop_sequences`, aby Claude přestal generovat, jakmile vydá váš požadovaný tag. To může ušetřit peníze a čas do posledního tokenu tím, že eliminuje závěrečné poznámky Claude poté, co vám již poskytl odpověď, na které vám záleží.

Pokud byste chtěli experimentovat s prompty lekce, aniž byste měnili jakýkoli obsah výše, přejděte až na konec poznámkového bloku lekce a navštivte [**Example Playground**](#example-playground).

---

## Cvičení
- [Cvičení 5.1 - Steph Curry GOAT](#exercise-51---steph-curry-goat)
- [Cvičení 5.2 - Dva Haiku](#exercise-52---two-haikus)
- [Cvičení 5.3 - Dva Haiku, Dva Zvířata](#exercise-53---two-haikus-two-animals)

### Cvičení 5.1 - Steph Curry GOAT
Přinucen k volbě, Claude označuje Michaela Jordana za nejlepšího basketbalistu všech dob. Můžeme přimět Clauda, aby vybral někoho jiného?

Změňte proměnnou `PREFILL` tak, aby **přiměla Clauda vytvořit podrobný argument, že nejlepším basketbalistou všech dob je Stephen Curry**. Snažte se neměnit nic jiného než `PREFILL`, protože to je hlavní zaměření tohoto cvičení.

In [ ]:
```python
# Šablona promptu s místem pro proměnnou content
PROMPT = f"Who is the best basketball player of all time? Please choose one specific player."

# Předvyplnění pro Claudeovu odpověď
PREFILL = ""

# Získání Claudeovy odpovědi
response = get_completion(PROMPT, prefill=PREFILL)

# Funkce pro hodnocení správnosti cvičení
def grade_exercise(text):
    return bool(re.search("Warrior", text))

# Výpis Claudeovy odpovědi
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))
```

❓ Pokud chcete nápovědu, spusťte buňku níže!

In [ ]:
print(hints.exercise_5_1_hint)

### Cvičení 5.2 - Dva haiku
Upravte níže uvedený `PROMPT` pomocí XML značek tak, aby Claude napsal dvě haiku o zvířeti místo jen jednoho. Mělo by být jasné, kde jeden básnický útvar končí a druhý začíná.

In [ ]:
```python
# Obsah proměnné
ANIMAL = "cats"

# Šablona promptu s místem pro obsah proměnné
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Předvyplnění pro odpověď Claude
PREFILL = "<haiku>"

# Získání odpovědi od Claude
response = get_completion(PROMPT, prefill=PREFILL)

# Funkce pro hodnocení správnosti cvičení
def grade_exercise(text):
    return bool(
        (re.search("cat", text.lower()) and re.search("<haiku>", text))
        and (text.count("\n") + 1) > 5
    )

# Výpis odpovědi od Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))
```

❓ Pokud chcete nápovědu, spusťte buňku níže!

In [ ]:
print(hints.exercise_5_2_hint)

### Cvičení 5.3 - Dva haiku, dvě zvířata
Upravte níže uvedený `PROMPT` tak, aby **Claude vytvořil dva haiku o dvou různých zvířatech**. Použijte `{ANIMAL1}` jako zástupný symbol pro první substituci a `{ANIMAL2}` jako zástupný symbol pro druhou substituci.

In [ ]:
```python
# První vstupní proměnná
ANIMAL1 = "Cat"

# Druhá vstupní proměnná
ANIMAL2 = "Dog"

# Šablona promptu s místem pro obsah proměnné
PROMPT = f"Please write a haiku about {ANIMAL1}. Put it in <haiku> tags."

# Získání odpovědi od Claude
response = get_completion(PROMPT)

# Funkce pro hodnocení správnosti cvičení
def grade_exercise(text):
    return bool(re.search("tail", text.lower()) and re.search("cat", text.lower()) and re.search("<haiku>", text))

# Výpis odpovědi od Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))
```

❓ Pokud chcete nápovědu, spusťte buňku níže!

In [ ]:
print(hints.exercise_5_3_hint)

### Gratulujeme!

Pokud jste vyřešili všechny úkoly až do tohoto bodu, jste připraveni přejít k další kapitole. Přejeme šťastné promptování!

---

## Příklad hřiště

Toto je prostor, kde můžete volně experimentovat s příklady promptů uvedenými v této lekci a upravovat prompty, abyste viděli, jak to může ovlivnit odpovědi Claude.

In [ ]:
```python
# Obsah proměnné
ANIMAL = "Rabbit"

# Šablona promptu s místem pro obsah proměnné
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Vytisknout odpověď od Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))
```

In [ ]:
```python
# Obsah proměnné
ANIMAL = "Cat"

# Šablona promptu s místem pro obsah proměnné
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Předvyplnění pro odpověď Claude
PREFILL = "<haiku>"

# Tisk odpovědi Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN:")
print(PROMPT)
print("\nASSISTANT TURN:")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))
```

In [ ]:
```python
# Obsah proměnné
ANIMAL = "Cat"

# Šablona promptu s místem pro obsah proměnné
PROMPT = f"Please write a haiku about {ANIMAL}. Use JSON format with the keys as \"first_line\", \"second_line\", and \"third_line\"."

# Předvyplnění pro odpověď Claude
PREFILL = "{"

# Tisk odpovědi Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))
```

In [ ]:
```python
# První vstupní proměnná
EMAIL = "Hi Zack, just pinging you for a quick update on that prompt you were supposed to write."

# Druhá vstupní proměnná
ADJECTIVE = "olde english"

# Šablona promptu s místem pro obsah proměnné
PROMPT = f"Hey Claude. Here is an email: <email>{EMAIL}</email>. Make this email more {ADJECTIVE}. Write the new version in <{ADJECTIVE}_email> XML tags."

# Předvyplnění pro Claudeovu odpověď (nyní jako f-string s proměnnou)
PREFILL = f"<{ADJECTIVE}_email>"

# Výpis Claudeovy odpovědi
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))
```